# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## Build retrieval evaluation ground-truth dataset

This notebook presents the methodology followed to construct a ground-truth dataset for evaluating the quality of the retrieval component.

To facilitate the manual identification of relevant chunks, a helper function was developed to retrieve and display all the chunks belonging to a given file, including their `chunk_id`. This allows the relevant chunks to be identified directly from the stored documents and used as ground truth for the evaluation.

The dataset was constructed using the [`spark-active-learning`](https://github.com/davidfernxndez/spark-active-learning) repository, with a **chunk size of 1000 characters** and a **chunk overlap of 200 characters**.

The notebook does not present the complete manual annotation process for every query and chunk. Instead, it illustrates the methodology and provides representative examples of how the evaluation dataset was constructed.

Importantly, the dataset was created **before testing the retrieval system**, ensuring that the ground truth is independent of the retrieval results.

### **Notebook objectives**

* Explain the methodology used to identify relevant chunks.
* Define the structure and purpose of the retrieval evaluation dataset.
* Show representative examples of the dataset construction process.

## Imports and Configuration

In [13]:
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.processor import Processor
from github_rag.chunking.chunker import Chunker
from github_rag.embedding.embedding_model import create_embedding_model
from github_rag.vectordatabase.chroma_store import ChromaStore
import chromadb
import time

To build the dataset, the same repository used for the demonstrations in the `notebooks/` directory is employed.

In [14]:
REPOSITORY_URL = "https://github.com/davidfernxndez/spark-active-learning"

## 1. Load vector database

This section checks whether the persistent ChromaDB collection already exists. If available, the existing database is loaded to avoid recomputing the embeddings; otherwise, the database is created by running the complete document processing pipeline.

In [15]:
def collection_exists(
    chroma_store: ChromaStore,
) -> bool:
    """Check whether the configured ChromaDB collection already exists."""

    # Connect to the persistent ChromaDB client.
    client = chromadb.PersistentClient(
        path=str(chroma_store.persist_directory)
    )

    # Check whether the configured collection is already stored locally.
    return chroma_store.collection_name in [
        collection.name
        for collection in client.list_collections()
    ]


# Configure the local ChromaDB vector store.
vector_store = ChromaStore(
    mode="local",
    collection_name="github_repo",
)

# Reuse the existing database when the collection is already available.
if collection_exists(vector_store):
    print("Existing local database found. Loading database...")

    # Load the persistent collection without recomputing embeddings.
    vector_store.load()

    # Check the number of documents stored in the collection.
    stored_documents = vector_store.count()
    print(f"Documents stored:   {stored_documents}")

else:
    print("Local database not found. Creating database...")

    # Initialize the GitHub repository loader.
    loader = GitHubRepositoryLoader(REPOSITORY_URL)

    # Load the repository files as LangChain Documents.
    documents = loader.load()

    print(f"Number of documents ingested: {len(documents)}")

    # Initialize the document processor.
    processor = Processor()

    # Apply repository-specific document processing.
    documents_processed = processor.process(documents)

    print(f"Number of documents processed: {len(documents_processed)}")

    # Initialize the chunker, which applies the appropriate
    # chunking strategy according to the document type.
    chunker = Chunker()

    # Split the processed documents into chunks.
    documents_chunking = chunker.chunk(documents_processed)

    print(f"Number of chunks generated: {len(documents_chunking)}")

    # Define the embedding model used to represent the chunks as vectors.
    embedding_model_name = (
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )

    # Load the embedding model.
    embedding_model = create_embedding_model(embedding_model_name)

    start_time = time.time()

    # Create the persistent ChromaDB collection and store the embeddings.
    vector_store.create(
        documents=documents_chunking,
        embedding_model=embedding_model
    )

    print(
        f"Database generation time: "
        f"{time.time() - start_time:.2f} seconds"
    )

    # Verify that all generated chunks were stored successfully.
    stored_documents = vector_store.count()

    print(f"Documents provided: {len(documents_chunking)}")
    print(f"Documents stored:   {stored_documents}")

Existing local database found. Loading database...
Documents stored:   183


## 2. Identify relevant chunks

To construct the ground-truth dataset, relevant chunks are identified by inspecting the contents of the files associated with each query. Given a `file_path`, all chunks belonging to that file are retrieved from the vector database and displayed in their original order, together with their `chunk_id`.

This allows the content of each chunk to be reviewed manually and the chunks containing the information required to answer a query to be selected as relevant.

In [16]:
def inspect_file_chunks(
    chroma_store: ChromaStore,
    file_path: str,
) -> None:
    """Display all chunks belonging to a specific file."""

    # Retrieve the documents whose file_path matches the requested file.
    results = chroma_store.get(
        filter={"file_path": file_path}
    )

    # Extract the chunk contents and their associated metadata.
    documents = results.get("documents") or []
    metadatas = results.get("metadatas") or []

    # Stop if no chunks are found for the requested file.
    if not documents:
        print(f"No chunks found for file: {file_path}")
        return

    # Sort the chunks according to their original position in the file.
    chunks = sorted(
        zip(documents, metadatas),
        key=lambda item: item[1].get("chunk_index", 0),
    )

    # Display basic information about the inspected file.
    print("=" * 100)
    print(f"FILE: {file_path}")
    print(f"NUMBER OF CHUNKS: {len(chunks)}")
    print("=" * 100)

    # Display each chunk together with its identifier and content.
    for index, (document, metadata) in enumerate(chunks, start=1):
        print()
        print("-" * 100)
        print(f"CHUNK {index}/{len(chunks)}")
        print(f"chunk_id: {metadata.get('chunk_id', 'N/A')}")
        print("-" * 100)
        print(document)

## 3. Retrieval evaluation dataset

The retrieval evaluation dataset defines the **ground truth** used to assess whether the retrieval component returns the chunks relevant to a given user query.

Each entry contains a user query and one or more `relevant_chunks`, identified by their `chunk_id`. Multiple chunks can be considered relevant when the information required to answer a query is distributed across different chunks.

The dataset follows the following structure:

```json
[
    {
        "query": "How is uncertainty estimated in the active learning process?",
        "relevant_chunks": [
            "scripts/AL_method.py::4",
            "scripts/AL_method.py::5"
        ]
    }
]
```

This structure allows the retrieved chunks to be compared against an independent set of relevant chunks when computing retrieval evaluation metrics.

Some examples are shown below:

#### Example query: ¿What clustering method is used?

The information needed to answer this question can be found in various files. For example, in the `diversity_k_means_selection()` function located in `scripts/AL_methods.py`, where it is specified that the algorithm is **BisectingKMeans**. The developed function is used to locate the specific chunk: 

In [17]:
inspect_file_chunks(vector_store, "scripts/AL_methods.py")

FILE: scripts/AL_methods.py
NUMBER OF CHUNKS: 27

----------------------------------------------------------------------------------------------------
CHUNK 1/27
chunk_id: scripts/AL_methods.py::0
----------------------------------------------------------------------------------------------------
"""
This module implements the methods required for the distributed Active Learning
workflow in Apache Spark, including model training and evaluation, uncertainty - 
and diversity-based instance selection and random selection.
"""

# ==============================================================================
# IMPORTS
# ==============================================================================

# PySpark SQL
import pyspark.sql.functions as sql_f

# PySpark ML - Features, Models & Evaluation
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import BisectingKMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.fu

By analyzing the content of the chunks, we can identify that the information needed to answer the question is found in the chunks: *scripts/AL_methods.py::15* and *scripts/AL_methods.py::18*. Therefore, the following query-chunk_id pair would be stored in the dataset:

```json
[
    {
        "query": "¿What clustering method is used?",
        "relevant_chunks": [
            "scripts/AL_methods.py::15",
            "scripts/AL_methods.py::18"
        ]
    }
]
```